<a href="https://colab.research.google.com/github/wangari2kimura/Special-topics-in-networking-/blob/main/CNS_4107_Lab_Network_Automation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### CNS 4107: Special Topics in Computer Networks - Network Automation
## Lab: Introduction to Network Automation with Python

---

**Objective:** By the end of this lab, you will be able to:
- Work with IP addresses and subnets programmatically using Python
- Resolve hostnames and probe open ports using the `socket` module
- Automate HTTP requests and monitor website availability
- Generate network device configurations automatically
- Parse and analyse simulated network device output
- Produce a structured network inventory report

**Estimated Time:** 90 – 120 minutes  
**Prerequisites:** Basic Python (functions, loops, dictionaries, f-strings)

---

>  **How to use this notebook:** Read each section, run every **Demo** cell in order, then complete the **Exercise** cells yourself. Write your answers to reflection questions in the provided Markdown cells.
>
> ⚠️ All exercises run entirely in Google Colab — no physical network hardware is required.

## 🔍 What is Network Automation?

**Network automation** is the process of using software to configure, manage, test, and operate network devices and services — replacing slow, error-prone manual work.

| Manual Task | Automated Equivalent |
|---|---|
| Logging into 50 routers to change a password | Script that SSHes into all devices at once |
| Checking if 200 servers are online | Automated ping/port sweep |
| Writing configs for 30 new switches | Template-based config generator |
| Reading interface status on every device | Parsed inventory report |
| Monitoring website uptime manually | Scheduled HTTP health-check script |

### Key Python Libraries for Network Automation

| Library | Purpose |
|---|---|
| `ipaddress` | IP address & subnet arithmetic (built-in) |
| `socket` | Low-level networking: DNS, TCP connections (built-in) |
| `requests` | HTTP requests & REST API automation |
| `paramiko` | SSH connections to network devices |
| `netmiko` | Multi-vendor SSH for routers & switches |
| `nmap` / `python-nmap` | Port scanning & host discovery |

In this introductory lab we focus on the first three, which are available in every Python environment.

---
## Part 0 — Environment Setup

Run the cell below once to install any extra libraries and import everything we need.

In [1]:
# Install third-party libraries (requests is pre-installed in Colab, listed here for clarity)
!pip install requests --quiet

# Standard library imports
import ipaddress
import socket
import os
import json
import csv
import time
import datetime
from pathlib import Path

# Third-party
import requests

# Create a directory for lab output files
LAB_DIR = Path("/content/network_lab")
LAB_DIR.mkdir(exist_ok=True)

print("Environment ready.")
print(f"Lab output directory: {LAB_DIR}")

Environment ready.
Lab output directory: /content/network_lab


---
## Part 1 — IP Address & Subnet Automation

Python's built-in `ipaddress` module lets you perform all the calculations a network engineer does by hand — but instantly and at scale.

### 1.1 Creating and Inspecting IP Address Objects

In [2]:
# --- Single IP address objects ---
ip4 = ipaddress.ip_address("192.168.1.45")
ip6 = ipaddress.ip_address("2001:db8::1")

print("=== IPv4 Address ===")
print(f"  Address      : {ip4}")
print(f"  Version      : IPv{ip4.version}")
print(f"  Is private?  : {ip4.is_private}")
print(f"  Is loopback? : {ip4.is_loopback}")
print(f"  Packed bytes : {ip4.packed}")

print()
print("=== IPv6 Address ===")
print(f"  Address      : {ip6}")
print(f"  Version      : IPv{ip6.version}")
print(f"  Compressed   : {ip6.compressed}")
print(f"  Exploded     : {ip6.exploded}")

=== IPv4 Address ===
  Address      : 192.168.1.45
  Version      : IPv4
  Is private?  : True
  Is loopback? : False
  Packed bytes : b'\xc0\xa8\x01-'

=== IPv6 Address ===
  Address      : 2001:db8::1
  Version      : IPv6
  Compressed   : 2001:db8::1
  Exploded     : 2001:0db8:0000:0000:0000:0000:0000:0001


### 1.2 Network / Subnet Objects

In [3]:
# Create a network object (strict=False allows host bits to be set)
net = ipaddress.ip_network("192.168.10.0/24", strict=False)

print("=== Subnet Info: 192.168.10.0/24 ===")
print(f"  Network address  : {net.network_address}")
print(f"  Broadcast address: {net.broadcast_address}")
print(f"  Subnet mask      : {net.netmask}")
print(f"  Wildcard mask    : {net.hostmask}")
print(f"  Prefix length    : /{net.prefixlen}")
print(f"  Total addresses  : {net.num_addresses}")
print(f"  Usable hosts     : {net.num_addresses - 2}")

# List the first 5 and last 5 usable host IPs
hosts = list(net.hosts())
print(f"\n  First 5 usable hosts: {[str(h) for h in hosts[:5]]}")
print(f"  Last  5 usable hosts: {[str(h) for h in hosts[-5:]]}")

=== Subnet Info: 192.168.10.0/24 ===
  Network address  : 192.168.10.0
  Broadcast address: 192.168.10.255
  Subnet mask      : 255.255.255.0
  Wildcard mask    : 0.0.0.255
  Prefix length    : /24
  Total addresses  : 256
  Usable hosts     : 254

  First 5 usable hosts: ['192.168.10.1', '192.168.10.2', '192.168.10.3', '192.168.10.4', '192.168.10.5']
  Last  5 usable hosts: ['192.168.10.250', '192.168.10.251', '192.168.10.252', '192.168.10.253', '192.168.10.254']


### 1.3 Automated Subnet Planning

Suppose you need to divide a large block of IPs into smaller subnets for different departments — automation does this instantly.

In [4]:
def plan_subnets(parent_network, prefix_len, department_names):
    """
    Splits parent_network into /prefix_len subnets and assigns
    each to a department name.
    """
    parent = ipaddress.ip_network(parent_network, strict=False)
    subnets = list(parent.subnets(new_prefix=prefix_len))

    if len(department_names) > len(subnets):
        raise ValueError("Not enough subnets for all departments!")

    print(f"\n📋 Subnet Plan  |  Parent: {parent}  →  /{prefix_len} subnets")
    print(f"{'Department':<20} {'Network':<20} {'Gateway':<18} {'Broadcast':<18} {'Usable Hosts'}")
    print("-" * 94)

    plan = []
    for dept, subnet in zip(department_names, subnets):
        hosts = list(subnet.hosts())
        gateway = str(hosts[0]) if hosts else "N/A"
        usable  = len(hosts)
        print(f"{dept:<20} {str(subnet):<20} {gateway:<18} {str(subnet.broadcast_address):<18} {usable}")
        plan.append({
            "department": dept,
            "network": str(subnet),
            "gateway": gateway,
            "broadcast": str(subnet.broadcast_address),
            "usable_hosts": usable
        })
    return plan


departments = ["Administration", "Finance", "ICT", "Library", "Student Labs"]
subnet_plan = plan_subnets("10.10.0.0/16", 24, departments)


📋 Subnet Plan  |  Parent: 10.10.0.0/16  →  /24 subnets
Department           Network              Gateway            Broadcast          Usable Hosts
----------------------------------------------------------------------------------------------
Administration       10.10.0.0/24         10.10.0.1          10.10.0.255        254
Finance              10.10.1.0/24         10.10.1.1          10.10.1.255        254
ICT                  10.10.2.0/24         10.10.2.1          10.10.2.255        254
Library              10.10.3.0/24         10.10.3.1          10.10.3.255        254
Student Labs         10.10.4.0/24         10.10.4.1          10.10.4.255        254


###  Exercise 1 — Subnet Membership Checker

Write a function `check_membership(ip_list, network_cidr)` that:
1. Takes a list of IP address strings and a network in CIDR notation.
2. For each IP, prints whether it is **inside** or **outside** the network.
3. Returns two lists: `inside` and `outside`.

**Hint:** Use the `in` operator — e.g., `ipaddress.ip_address(ip) in network`.

Test it with the data provided below.

In [5]:
#  Your code here
def check_membership(ip_list, network_cidr):
    network = ipaddress.ip_network(network_cidr, strict=False)  # create network object
    inside = []
    outside = []

    print(f"Network: {network}\n")
    print(f"{'IP Address':<20} {'Result'}")
    print("-" * 35)

    for ip in ip_list:
        addr = ipaddress.ip_address(ip)          # convert string to IP object
        if addr in network:                       # 'in' operator checks membership
            print(f"{ip:<20} ✅ INSIDE")
            inside.append(ip)
        else:
            print(f"{ip:<20} ❌ OUTSIDE")
            outside.append(ip)

    return inside, outside

# Test it
ip_list = [
    "192.168.1.1", "192.168.1.100", "192.168.2.5",
    "192.168.1.254", "10.0.0.1", "192.168.1.50"
]
network_cidr = "192.168.1.0/24"

inside, outside = check_membership(ip_list, network_cidr)
print(f"\nInside  ({len(inside)}): {inside}")
print(f"Outside ({len(outside)}): {outside}")

ip_list = [
    "192.168.1.1", "192.168.1.100", "192.168.2.5",
    "192.168.1.254", "10.0.0.1", "192.168.1.50"
]
network_cidr = "192.168.1.0/24"

def check_membership(ip_list, network_cidr):
    # TODO: implement this function
    pass

inside, outside = check_membership(ip_list, network_cidr) or ([], [])
print(f"\nInside  ({len(inside)}): {inside}")
print(f"Outside ({len(outside)}): {outside}")

Network: 192.168.1.0/24

IP Address           Result
-----------------------------------
192.168.1.1          ✅ INSIDE
192.168.1.100        ✅ INSIDE
192.168.2.5          ❌ OUTSIDE
192.168.1.254        ✅ INSIDE
10.0.0.1             ❌ OUTSIDE
192.168.1.50         ✅ INSIDE

Inside  (4): ['192.168.1.1', '192.168.1.100', '192.168.1.254', '192.168.1.50']
Outside (2): ['192.168.2.5', '10.0.0.1']

Inside  (0): []
Outside (0): []


---
## Part 2 — Hostname Resolution & Port Probing with `socket`

The `socket` module gives direct access to the OS networking stack. Network engineers use socket-based scripts to:
- Resolve DNS names to IPs
- Check whether a specific TCP port is open
- Build simple network scanners

### 2.1 DNS Resolution

In [6]:
def resolve_hostnames(hostnames):
    """Resolve a list of hostnames to IP addresses."""
    print(f"{'Hostname':<35} {'IP Address':<20} {'Status'}")
    print("-" * 65)
    results = {}
    for host in hostnames:
        try:
            ip = socket.gethostbyname(host)
            status = "✅ Resolved"
            results[host] = ip
        except socket.gaierror:
            ip = "N/A"
            status = "❌ Failed"
            results[host] = None
        print(f"{host:<35} {ip:<20} {status}")
    return results


hosts_to_resolve = [
    "google.com",
    "github.com",
    "strathmore.edu",
    "nonexistent.invalid",
    "wikipedia.org"
]

print("🔍 DNS Resolution Results")
dns_results = resolve_hostnames(hosts_to_resolve)

🔍 DNS Resolution Results
Hostname                            IP Address           Status
-----------------------------------------------------------------
google.com                          142.250.125.100      ✅ Resolved
github.com                          140.82.114.3         ✅ Resolved
strathmore.edu                      34.243.183.166       ✅ Resolved
nonexistent.invalid                 N/A                  ❌ Failed
wikipedia.org                       208.80.153.224       ✅ Resolved


### 2.2 TCP Port Prober

A port prober checks whether a service is listening on a given TCP port. This is the building block of network discovery tools.

In [7]:
def probe_port(host, port, timeout=2):
    """
    Try to open a TCP connection to host:port.
    Returns True if the port is open, False otherwise.
    """
    try:
        with socket.create_connection((host, port), timeout=timeout):
            return True
    except (socket.timeout, ConnectionRefusedError, OSError):
        return False


# Common port-to-service mapping
COMMON_PORTS = {
    21:  "FTP",
    22:  "SSH",
    25:  "SMTP",
    53:  "DNS",
    80:  "HTTP",
    110: "POP3",
    143: "IMAP",
    443: "HTTPS",
    3306:"MySQL",
    8080:"HTTP-Alt"
}

def scan_host(host, ports_dict):
    """Scan a host for a dictionary of {port: service_name} pairs."""
    print(f"\n🔎 Port Scan: {host}")
    print(f"{'Port':<8} {'Service':<12} {'Status'}")
    print("-" * 32)
    open_ports = []
    for port, service in ports_dict.items():
        is_open = probe_port(host, port)
        status = "🟢 OPEN" if is_open else "🔴 CLOSED"
        print(f"{port:<8} {service:<12} {status}")
        if is_open:
            open_ports.append(port)
    print(f"\n  Open ports found: {open_ports if open_ports else 'None'}")
    return open_ports

open_ports = scan_host("google.com", COMMON_PORTS)


🔎 Port Scan: google.com
Port     Service      Status
--------------------------------
21       FTP          🔴 CLOSED
22       SSH          🔴 CLOSED
25       SMTP         🔴 CLOSED
53       DNS          🔴 CLOSED
80       HTTP         🟢 OPEN
110      POP3         🔴 CLOSED
143      IMAP         🔴 CLOSED
443      HTTPS        🟢 OPEN
3306     MySQL        🔴 CLOSED
8080     HTTP-Alt     🔴 CLOSED

  Open ports found: [80, 443]


### 2.3 Automated Multi-Host Port Report

In [8]:
def multi_host_report(hosts, ports_dict):
    """Scan multiple hosts and return a structured report."""
    report = {}
    for host in hosts:
        ip = dns_results.get(host) or "unresolved"
        open_p = scan_host(host, ports_dict)
        report[host] = {"ip": ip, "open_ports": open_p}
    return report

# Scan only HTTP/HTTPS for speed in this demo
web_ports = {80: "HTTP", 443: "HTTPS"}
scan_report = multi_host_report(["google.com", "github.com", "wikipedia.org"], web_ports)

print("\n📊 Summary Report")
print(json.dumps(scan_report, indent=2))


🔎 Port Scan: google.com
Port     Service      Status
--------------------------------
80       HTTP         🟢 OPEN
443      HTTPS        🟢 OPEN

  Open ports found: [80, 443]

🔎 Port Scan: github.com
Port     Service      Status
--------------------------------
80       HTTP         🟢 OPEN
443      HTTPS        🟢 OPEN

  Open ports found: [80, 443]

🔎 Port Scan: wikipedia.org
Port     Service      Status
--------------------------------
80       HTTP         🟢 OPEN
443      HTTPS        🟢 OPEN

  Open ports found: [80, 443]

📊 Summary Report
{
  "google.com": {
    "ip": "142.250.125.100",
    "open_ports": [
      80,
      443
    ]
  },
  "github.com": {
    "ip": "140.82.114.3",
    "open_ports": [
      80,
      443
    ]
  },
  "wikipedia.org": {
    "ip": "208.80.153.224",
    "open_ports": [
      80,
      443
    ]
  }
}


### ✏️ Exercise 2 — Reverse DNS Lookup

Write a function `reverse_lookup(ip_list)` that:
1. Takes a list of IP address strings.
2. Uses `socket.gethostbyaddr(ip)` to find the hostname for each IP.
3. Prints a table showing each IP and its resolved hostname (or `"No PTR record"` if lookup fails).
4. Returns a dictionary `{ip: hostname_or_None}`.

**Hint:** `socket.gethostbyaddr()` raises `socket.herror` on failure.

In [9]:
# ✏️ Your code here

def check_membership(ip_list, network_cidr):
    network = ipaddress.ip_network(network_cidr, strict=False)  # create network object
    inside = []
    outside = []

    print(f"Network: {network}\n")
    print(f"{'IP Address':<20} {'Result'}")
    print("-" * 35)

    for ip in ip_list:
        addr = ipaddress.ip_address(ip)          # convert string to IP object
        if addr in network:                       # 'in' operator checks membership
            print(f"{ip:<20} ✅ INSIDE")
            inside.append(ip)
        else:
            print(f"{ip:<20} ❌ OUTSIDE")
            outside.append(ip)

    return inside, outside

# Test it
ip_list = [
    "192.168.1.1", "192.168.1.100", "192.168.2.5",
    "192.168.1.254", "10.0.0.1", "192.168.1.50"
]
network_cidr = "192.168.1.0/24"

inside, outside = check_membership(ip_list, network_cidr)
print(f"\nInside  ({len(inside)}): {inside}")
print(f"Outside ({len(outside)}): {outside}")

ips_to_reverse = ["8.8.8.8", "1.1.1.1", "208.67.222.222", "192.168.1.1"]

def reverse_lookup(ip_list):
    # TODO: implement this function
    pass

reverse_lookup(ips_to_reverse)

Network: 192.168.1.0/24

IP Address           Result
-----------------------------------
192.168.1.1          ✅ INSIDE
192.168.1.100        ✅ INSIDE
192.168.2.5          ❌ OUTSIDE
192.168.1.254        ✅ INSIDE
10.0.0.1             ❌ OUTSIDE
192.168.1.50         ✅ INSIDE

Inside  (4): ['192.168.1.1', '192.168.1.100', '192.168.1.254', '192.168.1.50']
Outside (2): ['192.168.2.5', '10.0.0.1']


---
## Part 3 — HTTP Automation with `requests`

The `requests` library lets you interact with web servers and REST APIs — essential for modern network automation where devices expose HTTP/REST management interfaces.

### 3.1 Website Health Checker

In [10]:
def check_website_health(urls, timeout=5):
    """
    Check HTTP/HTTPS availability for a list of URLs.
    Returns a list of result dictionaries.
    """
    results = []
    print(f"{'URL':<40} {'Status Code':<14} {'Response (ms)':<16} {'Health'}")
    print("-" * 82)

    for url in urls:
        start = time.time()
        try:
            resp = requests.get(url, timeout=timeout, allow_redirects=True)
            elapsed_ms = round((time.time() - start) * 1000)
            code = resp.status_code
            health = "✅ UP" if code < 400 else "⚠️  DEGRADED"
        except requests.exceptions.ConnectionError:
            code, elapsed_ms, health = "ERR", 0, "❌ DOWN"
        except requests.exceptions.Timeout:
            code, elapsed_ms, health = "TIMEOUT", timeout * 1000, "⏱️  TIMEOUT"
        except Exception as e:
            code, elapsed_ms, health = "ERR", 0, f"❌ {type(e).__name__}"

        print(f"{url:<40} {str(code):<14} {str(elapsed_ms) + ' ms':<16} {health}")
        results.append({"url": url, "status_code": code, "response_ms": elapsed_ms, "health": health})

    up = sum(1 for r in results if "UP" in r["health"])
    print(f"\n  Summary: {up}/{len(results)} sites are UP")
    return results


sites = [
    "https://www.google.com",
    "https://www.github.com",
    "https://httpstat.us/200",
    "https://httpstat.us/404",
    "https://httpstat.us/503"
]

health_results = check_website_health(sites)

URL                                      Status Code    Response (ms)    Health
----------------------------------------------------------------------------------
https://www.google.com                   200            79 ms            ✅ UP
https://www.github.com                   200            272 ms           ✅ UP
https://httpstat.us/200                  ERR            0 ms             ❌ DOWN
https://httpstat.us/404                  ERR            0 ms             ❌ DOWN
https://httpstat.us/503                  ERR            0 ms             ❌ DOWN

  Summary: 2/5 sites are UP


### 3.2 Querying a Public REST API

Network automation often involves querying REST APIs — for example, to get IP geolocation, check BGP data, or retrieve device status from an NMS.

In [11]:
def geolocate_ip(ip_address):
    """
    Query the free ip-api.com REST endpoint to get geolocation
    data for a given public IP address.
    """
    url = f"http://ip-api.com/json/{ip_address}"
    try:
        response = requests.get(url, timeout=5)
        response.raise_for_status()
        data = response.json()

        if data.get("status") == "success":
            return {
                "ip":          data.get("query"),
                "country":     data.get("country"),
                "region":      data.get("regionName"),
                "city":        data.get("city"),
                "isp":         data.get("isp"),
                "org":         data.get("org"),
                "latitude":    data.get("lat"),
                "longitude":   data.get("lon"),
                "timezone":    data.get("timezone")
            }
        else:
            return {"error": data.get("message", "Unknown error")}
    except requests.RequestException as e:
        return {"error": str(e)}


# Geolocate a set of well-known public IPs
public_ips = ["8.8.8.8", "1.1.1.1", "208.67.222.222"]

print("🌍 IP Geolocation Results\n")
for ip in public_ips:
    info = geolocate_ip(ip)
    print(f"  IP: {ip}")
    if "error" in info:
        print(f"    ❌ Error: {info['error']}")
    else:
        for key, value in info.items():
            if key != "ip":
                print(f"    {key:<12}: {value}")
    print()

🌍 IP Geolocation Results

  IP: 8.8.8.8
    country     : United States
    region      : Virginia
    city        : Ashburn
    isp         : Google LLC
    org         : Google Public DNS
    latitude    : 39.03
    longitude   : -77.5
    timezone    : America/New_York

  IP: 1.1.1.1
    country     : Australia
    region      : Queensland
    city        : South Brisbane
    isp         : Cloudflare, Inc
    org         : APNIC and Cloudflare DNS Resolver project
    latitude    : -27.4766
    longitude   : 153.0166
    timezone    : Australia/Brisbane

  IP: 208.67.222.222
    country     : United States
    region      : California
    city        : San Jose
    isp         : Cisco OpenDNS, LLC
    org         : Cisco OpenDNS, LLC
    latitude    : 37.4084
    longitude   : -121.954
    timezone    : America/Los_Angeles



### Exercise 3 — Automated Uptime Logger

Build on `check_website_health()` to write a function `uptime_logger(urls, interval_seconds, checks)` that:
1. Runs `checks` rounds of health checks, pausing `interval_seconds` between each round.
2. Appends each result to a list with a timestamp.
3. After all rounds, prints a final summary table showing:
   - Each URL
   - Total checks performed
   - Number of times it was UP
   - Uptime percentage
4. Saves the full log as a JSON file in `LAB_DIR`.

**Use:** `checks=3`, `interval_seconds=2` for a quick test.

In [12]:
#  Your code here

def uptime_logger(urls, interval_seconds=2, checks=3):
    all_logs = []      # stores every single check result with timestamp
    # tracker: how many times each URL was UP
    up_counts = {url: 0 for url in urls}

    for round_num in range(1, checks + 1):
        print(f"\n{'='*50}")
        print(f"🔄 Round {round_num} of {checks}  —  {datetime.datetime.now().strftime('%H:%M:%S')}")
        print(f"{'='*50}")

        round_results = check_website_health(urls)  # reuse the demo function!

        for result in round_results:
            result["round"] = round_num
            result["timestamp"] = datetime.datetime.now().isoformat()
            all_logs.append(result)

            if "UP" in result["health"]:            # count successful checks
                up_counts[result["url"]] += 1

        if round_num < checks:                      # don't sleep after last round
            print(f"\n⏳ Waiting {interval_seconds}s before next round...")
            time.sleep(interval_seconds)

    # --- Final Summary Table ---
    print(f"\n{'='*50}")
    print("📊 UPTIME SUMMARY")
    print(f"{'='*50}")
    print(f"{'URL':<40} {'Checks':<8} {'UP':<6} {'Uptime %'}")
    print("-" * 62)

    for url in urls:
        up = up_counts[url]
        pct = round((up / checks) * 100, 1)
        print(f"{url:<40} {checks:<8} {up:<6} {pct}%")

    # --- Save JSON log ---
    log_path = LAB_DIR / "uptime_log.json"
    with open(log_path, "w") as f:
        json.dump(all_logs, f, indent=2)
    print(f"\n💾 Full log saved to: {log_path}")

    return all_logs


# Run it
uptime_logger(
    urls=["https://www.google.com", "https://httpstat.us/200", "https://httpstat.us/503"],
    interval_seconds=2,
    checks=3
)

def uptime_logger(urls, interval_seconds=2, checks=3):
    # TODO: implement this function
    pass

uptime_logger(
    urls=["https://www.google.com", "https://httpstat.us/200", "https://httpstat.us/503"],
    interval_seconds=2,
    checks=3
)


🔄 Round 1 of 3  —  05:26:03
URL                                      Status Code    Response (ms)    Health
----------------------------------------------------------------------------------
https://www.google.com                   200            68 ms            ✅ UP
https://httpstat.us/200                  ERR            0 ms             ❌ DOWN
https://httpstat.us/503                  ERR            0 ms             ❌ DOWN

  Summary: 1/3 sites are UP

⏳ Waiting 2s before next round...

🔄 Round 2 of 3  —  05:26:15
URL                                      Status Code    Response (ms)    Health
----------------------------------------------------------------------------------
https://www.google.com                   200            66 ms            ✅ UP
https://httpstat.us/200                  ERR            0 ms             ❌ DOWN
https://httpstat.us/503                  ERR            0 ms             ❌ DOWN

  Summary: 1/3 sites are UP

⏳ Waiting 2s before next round...

🔄 Round 3 o

---
## Part 4 — Automated Network Device Config Generation

One of the most impactful uses of network automation is **configuration generation** — producing consistent, error-free device configs from a template and a data source.

We will simulate generating Cisco IOS-style configurations for a small campus network.

### 4.1 Defining Device Data

In [13]:
# Device inventory — in a real scenario this could come from a CSV, JSON, or CMDB
DEVICES = [
    {
        "hostname":     "SW-ADMIN-01",
        "device_type":  "switch",
        "location":     "Admin Block",
        "mgmt_ip":      "10.10.1.2",
        "mgmt_mask":    "255.255.255.0",
        "default_gw":   "10.10.1.1",
        "vlans":        [{"id": 10, "name": "ADMIN"}, {"id": 99, "name": "MGMT"}],
        "ntp_server":   "10.10.0.1",
        "dns_server":   "8.8.8.8"
    },
    {
        "hostname":     "SW-ICT-01",
        "device_type":  "switch",
        "location":     "ICT Block",
        "mgmt_ip":      "10.10.2.2",
        "mgmt_mask":    "255.255.255.0",
        "default_gw":   "10.10.2.1",
        "vlans":        [{"id": 20, "name": "ICT"}, {"id": 99, "name": "MGMT"}],
        "ntp_server":   "10.10.0.1",
        "dns_server":   "8.8.8.8"
    },
    {
        "hostname":     "RTR-CORE-01",
        "device_type":  "router",
        "location":     "Server Room",
        "mgmt_ip":      "10.10.0.1",
        "mgmt_mask":    "255.255.255.0",
        "default_gw":   "10.10.0.254",
        "vlans":        [],
        "ntp_server":   "pool.ntp.org",
        "dns_server":   "8.8.8.8"
    }
]

print(f"📋 Loaded {len(DEVICES)} devices into inventory.")
for d in DEVICES:
    print(f"   [{d['device_type'].upper()}] {d['hostname']}  —  {d['mgmt_ip']}  —  {d['location']}")

📋 Loaded 3 devices into inventory.
   [SWITCH] SW-ADMIN-01  —  10.10.1.2  —  Admin Block
   [SWITCH] SW-ICT-01  —  10.10.2.2  —  ICT Block
   [ROUTER] RTR-CORE-01  —  10.10.0.1  —  Server Room


### 4.2 Config Template & Generator

In [14]:
def generate_base_config(device):
    """
    Generate a Cisco IOS-style base configuration string
    from a device dictionary.
    """
    timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    lines = []

    # Header
    lines += [
        "! =" * 20,
        f"! Device  : {device['hostname']}",
        f"! Type    : {device['device_type'].title()}",
        f"! Location: {device['location']}",
        f"! Generated: {timestamp}",
        "! =" * 20,
        "",
        "version 15.2",
        "no service timestamps log",
        "no service timestamps debug",
        "no service password-encryption",
        ""
    ]

    # Hostname
    lines += [
        f"hostname {device['hostname']}",
        ""
    ]

    # Management interface
    if device["device_type"] == "switch":
        lines += [
            "interface Vlan99",
            f" description Management Interface",
            f" ip address {device['mgmt_ip']} {device['mgmt_mask']}",
            " no shutdown",
            "",
            f"ip default-gateway {device['default_gw']}",
            ""
        ]
    else:
        lines += [
            "interface GigabitEthernet0/0",
            f" description Management Interface — {device['location']}",
            f" ip address {device['mgmt_ip']} {device['mgmt_mask']}",
            " duplex auto",
            " speed auto",
            " no shutdown",
            "",
            f"ip route 0.0.0.0 0.0.0.0 {device['default_gw']}",
            ""
        ]

    # VLANs (switches only)
    if device["vlans"]:
        for vlan in device["vlans"]:
            lines += [
                f"vlan {vlan['id']}",
                f" name {vlan['name']}",
                ""
            ]

    # NTP & DNS
    lines += [
        f"ntp server {device['ntp_server']}",
        f"ip name-server {device['dns_server']}",
        ""
    ]

    # SSH & security
    lines += [
        "ip domain-name strathmore.local",
        "crypto key generate rsa modulus 2048",
        "ip ssh version 2",
        "",
        "line vty 0 4",
        " login local",
        " transport input ssh",
        "",
        "line console 0",
        " logging synchronous",
        "",
        "end"
    ]

    return "\n".join(lines)


# Generate and display config for the first device
sample_config = generate_base_config(DEVICES[0])
print(sample_config)

! =! =! =! =! =! =! =! =! =! =! =! =! =! =! =! =! =! =! =! =
! Device  : SW-ADMIN-01
! Type    : Switch
! Location: Admin Block
! Generated: 2026-05-05 05:26:30
! =! =! =! =! =! =! =! =! =! =! =! =! =! =! =! =! =! =! =! =

version 15.2
no service timestamps log
no service timestamps debug
no service password-encryption

hostname SW-ADMIN-01

interface Vlan99
 description Management Interface
 ip address 10.10.1.2 255.255.255.0
 no shutdown

ip default-gateway 10.10.1.1

vlan 10
 name ADMIN

vlan 99
 name MGMT

ntp server 10.10.0.1
ip name-server 8.8.8.8

ip domain-name strathmore.local
crypto key generate rsa modulus 2048
ip ssh version 2

line vty 0 4
 login local
 transport input ssh

line console 0
 logging synchronous

end


### 4.3 Bulk Config Generation & Save to Files

In [15]:
def generate_all_configs(devices, output_dir):
    """Generate configs for every device and save each to a .txt file."""
    cfg_dir = Path(output_dir) / "configs"
    cfg_dir.mkdir(exist_ok=True)

    print(f"⚙️  Generating configs into: {cfg_dir}\n")
    manifest = []

    for device in devices:
        config = generate_base_config(device)
        filename = f"{device['hostname'].lower()}_config.txt"
        filepath = cfg_dir / filename

        with open(filepath, "w") as f:
            f.write(config)

        size_kb = round(filepath.stat().st_size / 1024, 2)
        print(f"  ✅ {filename:<35}  ({size_kb} KB)")
        manifest.append({"device": device["hostname"], "file": filename, "size_kb": size_kb})

    print(f"\n  {len(manifest)} config files saved.")
    return manifest

config_manifest = generate_all_configs(DEVICES, LAB_DIR)

⚙️  Generating configs into: /content/network_lab/configs

  ✅ sw-admin-01_config.txt               (0.72 KB)
  ✅ sw-ict-01_config.txt                 (0.71 KB)
  ✅ rtr-core-01_config.txt               (0.74 KB)

  3 config files saved.


###  Exercise 4 — VLAN Config Extender

Write a function `add_access_port_config(device, port_assignments)` that:
1. Takes a device dictionary and a list of `{"interface": "Fa0/1", "vlan": 10, "description": "PC-Admin"}` dictionaries.
2. Generates Cisco IOS-style access port configuration lines for each assignment.
3. Prints the generated lines.
4. Saves the output to `<hostname>_ports_config.txt` in `LAB_DIR`.

**Expected output per interface:**
```
interface FastEthernet0/1
 description PC-Admin
 switchport mode access
 switchport access vlan 10
 no shutdown
```

In [23]:
#  Your code here

def add_access_port_config(device, port_assignments):
    lines = []
    timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    # Header comment block
    lines += [
        f"! Access Port Config  —  {device['hostname']}",
        f"! Generated: {timestamp}",
        f"! Location : {device['location']}",
        "",
    ]

    for port in port_assignments:
        lines += [
            f"interface FastEthernet{port['interface'].replace('Fa', '')}",
            f" description {port['description']}",
            f" switchport mode access",
            f" switchport access vlan {port['vlan']}",
            f" no shutdown",
            "",                                     # blank line between interfaces
        ]

    config_text = "\n".join(lines)

    # Print to screen
    print(config_text)

    # Save to file
    filename = f"{device['hostname'].lower()}_ports_config.txt"
    filepath = LAB_DIR / filename
    with open(filepath, "w") as f:
        f.write(config_text)

    print(f"💾 Saved to: {filepath}")
    return config_text


# Test it
port_assignments = [
    {"interface": "Fa0/1",  "vlan": 10, "description": "PC-Admin-01"},
    {"interface": "Fa0/2",  "vlan": 10, "description": "PC-Admin-02"},
    {"interface": "Fa0/24", "vlan": 99, "description": "MGMT-Uplink"}
]

add_access_port_config(DEVICES[0], port_assignments)

port_assignments = [
    {"interface": "Fa0/1",  "vlan": 10, "description": "PC-Admin-01"},
    {"interface": "Fa0/2",  "vlan": 10, "description": "PC-Admin-02"},
    {"interface": "Fa0/24", "vlan": 99, "description": "MGMT-Uplink"}
]

def add_access_port_config(device, port_assignments):
    # TODO: implement this function
    pass

add_access_port_config(DEVICES[0], port_assignments)

! Access Port Config  —  SW-ADMIN-01
! Generated: 2026-05-05 05:48:00
! Location : Admin Block

interface FastEthernet0/1
 description PC-Admin-01
 switchport mode access
 switchport access vlan 10
 no shutdown

interface FastEthernet0/2
 description PC-Admin-02
 switchport mode access
 switchport access vlan 10
 no shutdown

interface FastEthernet0/24
 description MGMT-Uplink
 switchport mode access
 switchport access vlan 99
 no shutdown

💾 Saved to: /content/network_lab/sw-admin-01_ports_config.txt


---
## Part 5 — Parsing Simulated Network Output

In real networks, automation scripts often SSH into devices, run commands like `show ip interface brief`, and **parse the text output** to extract structured data. Here we simulate that workflow.

### 5.1 Simulated `show ip interface brief` Output

In [24]:
# This is what a real Cisco device would return over SSH
SHOW_IP_INT_BRIEF = """\
Interface              IP-Address      OK? Method Status                Protocol
GigabitEthernet0/0     10.10.0.1       YES NVRAM  up                    up
GigabitEthernet0/1     10.10.1.1       YES NVRAM  up                    up
GigabitEthernet0/2     10.10.2.1       YES NVRAM  up                    up
GigabitEthernet0/3     unassigned      YES NVRAM  administratively down down
Loopback0              1.1.1.1         YES NVRAM  up                    up
Vlan10                 192.168.10.1    YES NVRAM  up                    up
Vlan20                 192.168.20.1    YES NVRAM  up                    up
Vlan99                 10.99.0.1       YES NVRAM  up                    up
"""

print(SHOW_IP_INT_BRIEF)

Interface              IP-Address      OK? Method Status                Protocol
GigabitEthernet0/0     10.10.0.1       YES NVRAM  up                    up
GigabitEthernet0/1     10.10.1.1       YES NVRAM  up                    up
GigabitEthernet0/2     10.10.2.1       YES NVRAM  up                    up
GigabitEthernet0/3     unassigned      YES NVRAM  administratively down down
Loopback0              1.1.1.1         YES NVRAM  up                    up
Vlan10                 192.168.10.1    YES NVRAM  up                    up
Vlan20                 192.168.20.1    YES NVRAM  up                    up
Vlan99                 10.99.0.1       YES NVRAM  up                    up



### 5.2 Parsing the Output into Structured Data

In [25]:
import re

def parse_show_ip_int_brief(raw_output):
    """
    Parse Cisco 'show ip interface brief' text output
    into a list of structured dictionaries.
    """
    interfaces = []
    # Regex: capture interface, IP, status, protocol columns
    pattern = re.compile(
        r"^(\S+)\s+(\S+)\s+\S+\s+\S+\s+(\S+(?:\s+\S+)?)\s+(\S+)$",
        re.MULTILINE
    )

    for match in pattern.finditer(raw_output):
        interface, ip, status, protocol = match.groups()
        if interface.lower() == "interface":  # skip header line
            continue
        interfaces.append({
            "interface": interface,
            "ip_address": ip,
            "status":    status.strip(),
            "protocol":  protocol.strip()
        })

    return interfaces


parsed = parse_show_ip_int_brief(SHOW_IP_INT_BRIEF)

print(f"{'Interface':<25} {'IP Address':<18} {'Status':<26} {'Protocol'}")
print("-" * 80)
for intf in parsed:
    status_icon = "🟢" if intf["status"] == "up" else "🔴"
    print(f"{intf['interface']:<25} {intf['ip_address']:<18} {status_icon} {intf['status']:<23} {intf['protocol']}")

up_count   = sum(1 for i in parsed if i["status"] == "up")
down_count = len(parsed) - up_count
print(f"\n  🟢 UP: {up_count}   🔴 DOWN/ADMIN-DOWN: {down_count}")

Interface                 IP Address         Status                     Protocol
--------------------------------------------------------------------------------
GigabitEthernet0/0        10.10.0.1          🟢 up                      up
GigabitEthernet0/1        10.10.1.1          🟢 up                      up
GigabitEthernet0/2        10.10.2.1          🟢 up                      up
GigabitEthernet0/3        unassigned         🔴 administratively down   down
Loopback0                 1.1.1.1            🟢 up                      up
Vlan10                    192.168.10.1       🟢 up                      up
Vlan20                    192.168.20.1       🟢 up                      up
Vlan99                    10.99.0.1          🟢 up                      up

  🟢 UP: 7   🔴 DOWN/ADMIN-DOWN: 1


### ✏️ Exercise 5 — Interface Anomaly Detector

Using the `parsed` list from above, write a function `detect_anomalies(interfaces)` that:
1. Flags any interface where `status` is **up** but `protocol` is **down** (a common misconfiguration indicator).
2. Flags any interface that is `administratively down`.
3. Flags any interface with `unassigned` as its IP address while having `status == up`.
4. Prints a clear anomaly report. If no anomalies are found, print `"No anomalies detected"`.

Then test it against the simulated data *and* against a new list you design that includes at least one of each anomaly type.

In [26]:
# ✏️ Your code here

def detect_anomalies(interfaces):
    anomalies = []

    for intf in interfaces:
        name     = intf["interface"]
        ip       = intf["ip_address"]
        status   = intf["status"]
        protocol = intf["protocol"]

        # Anomaly Type 1: Up but protocol is down (layer 1 OK, layer 2 broken)
        if status == "up" and protocol == "down":
            anomalies.append({
                "interface": name,
                "type": "⚠️  STATUS/PROTOCOL MISMATCH",
                "detail": f"Status is UP but protocol is DOWN"
            })

        # Anomaly Type 2: Administratively shut down
        elif "administratively" in status:
            anomalies.append({
                "interface": name,
                "type": "🔴 ADMIN DOWN",
                "detail": f"Interface has been manually shut down"
            })

        # Anomaly Type 3: IP unassigned but status is up
        elif ip == "unassigned" and status == "up":
            anomalies.append({
                "interface": name,
                "type": "⚠️  NO IP ADDRESS",
                "detail": f"Interface is UP but has no IP assigned"
            })

    # Print report
    print("🔍 ANOMALY DETECTION REPORT")
    print("=" * 55)

    if not anomalies:
        print("✅ No anomalies detected")
    else:
        print(f"Found {len(anomalies)} anomaly(s):\n")
        for a in anomalies:
            print(f"  Interface : {a['interface']}")
            print(f"  Type      : {a['type']}")
            print(f"  Detail    : {a['detail']}")
            print("-" * 55)

    return anomalies


# Test 1 — against the simulated data from the demo
print("TEST 1: Simulated device output")
detect_anomalies(parsed)

# Test 2 — custom list with one of each anomaly type
print("\nTEST 2: Custom list with all three anomaly types")
custom_interfaces = [
    {"interface": "GigabitEthernet0/0", "ip_address": "10.0.0.1",    "status": "up",                   "protocol": "up"},
    {"interface": "GigabitEthernet0/1", "ip_address": "10.0.0.2",    "status": "up",                   "protocol": "down"},   # Type 1
    {"interface": "GigabitEthernet0/2", "ip_address": "unassigned",  "status": "administratively down","protocol": "down"},   # Type 2
    {"interface": "GigabitEthernet0/3", "ip_address": "unassigned",  "status": "up",                   "protocol": "up"}      # Type 3
]
detect_anomalies(custom_interfaces)


TEST 1: Simulated device output
🔍 ANOMALY DETECTION REPORT
Found 1 anomaly(s):

  Interface : GigabitEthernet0/3
  Type      : 🔴 ADMIN DOWN
  Detail    : Interface has been manually shut down
-------------------------------------------------------

TEST 2: Custom list with all three anomaly types
🔍 ANOMALY DETECTION REPORT
Found 3 anomaly(s):

  Interface : GigabitEthernet0/1
  Type      : ⚠️  STATUS/PROTOCOL MISMATCH
  Detail    : Status is UP but protocol is DOWN
-------------------------------------------------------
  Interface : GigabitEthernet0/2
  Type      : 🔴 ADMIN DOWN
  Detail    : Interface has been manually shut down
-------------------------------------------------------
  Interface : GigabitEthernet0/3
  Type      : ⚠️  NO IP ADDRESS
  Detail    : Interface is UP but has no IP assigned
-------------------------------------------------------


[{'interface': 'GigabitEthernet0/1',
  'type': '⚠️  STATUS/PROTOCOL MISMATCH',
  'detail': 'Status is UP but protocol is DOWN'},
 {'interface': 'GigabitEthernet0/2',
  'type': '🔴 ADMIN DOWN',
  'detail': 'Interface has been manually shut down'},
 {'interface': 'GigabitEthernet0/3',
  'type': '⚠️  NO IP ADDRESS',
  'detail': 'Interface is UP but has no IP assigned'}]

---
## Part 6 — Generating a Network Inventory Report

Bringing everything together: auto-generate a complete inventory report in both CSV and JSON formats.

### 6.1 Building the Inventory

In [27]:
def build_inventory(devices, interface_data, health_results):
    """
    Combine device, interface and health data into a unified inventory.
    """
    health_map = {r["url"]: r for r in health_results}

    inventory = []
    for device in devices:
        # Count up interfaces
        up_intfs  = sum(1 for i in interface_data if i["status"] == "up")
        all_intfs = len(interface_data)

        record = {
            "hostname":          device["hostname"],
            "type":              device["device_type"],
            "location":          device["location"],
            "management_ip":     device["mgmt_ip"],
            "vlans":             len(device["vlans"]),
            "interfaces_up":     up_intfs,
            "interfaces_total":  all_intfs,
            "ntp_server":        device["ntp_server"],
            "generated_at":      datetime.datetime.now().isoformat()
        }
        inventory.append(record)

    return inventory

inventory = build_inventory(DEVICES, parsed, health_results)

print("📦 Inventory built:")
for item in inventory:
    print(f"  {item['hostname']:<15} {item['type']:<8} {item['management_ip']:<16} VLANs:{item['vlans']}  Intf Up:{item['interfaces_up']}/{item['interfaces_total']}")

📦 Inventory built:
  SW-ADMIN-01     switch   10.10.1.2        VLANs:2  Intf Up:7/8
  SW-ICT-01       switch   10.10.2.2        VLANs:2  Intf Up:7/8
  RTR-CORE-01     router   10.10.0.1        VLANs:0  Intf Up:7/8


### 6.2 Exporting the Report

In [28]:
def export_inventory(inventory, output_dir):
    """Export inventory to both JSON and CSV formats."""
    out = Path(output_dir)

    # --- JSON ---
    json_path = out / "network_inventory.json"
    with open(json_path, "w") as f:
        json.dump(inventory, f, indent=2)
    print(f"✅ JSON report saved : {json_path}")

    # --- CSV ---
    csv_path = out / "network_inventory.csv"
    if inventory:
        with open(csv_path, "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=inventory[0].keys())
            writer.writeheader()
            writer.writerows(inventory)
    print(f"✅ CSV  report saved : {csv_path}")

    # --- Plain-text summary ---
    txt_path = out / "network_inventory_summary.txt"
    with open(txt_path, "w") as f:
        f.write("NETWORK INVENTORY REPORT\n")
        f.write("=" * 60 + "\n")
        f.write(f"Generated: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write(f"Total devices: {len(inventory)}\n\n")
        for item in inventory:
            f.write(f"  Hostname   : {item['hostname']}\n")
            f.write(f"  Type       : {item['type']}\n")
            f.write(f"  Location   : {item['location']}\n")
            f.write(f"  Mgmt IP    : {item['management_ip']}\n")
            f.write(f"  VLANs      : {item['vlans']}\n")
            f.write(f"  Interfaces : {item['interfaces_up']}/{item['interfaces_total']} UP\n")
            f.write("-" * 40 + "\n")
    print(f"✅ TXT  report saved : {txt_path}")


export_inventory(inventory, LAB_DIR)

# List all files generated during the lab
print("\n📁 All files in lab directory:")
for p in sorted(LAB_DIR.rglob("*")):
    if p.is_file():
        size = round(p.stat().st_size / 1024, 2)
        print(f"   {str(p.relative_to(LAB_DIR)):<45} {size} KB")

✅ JSON report saved : /content/network_lab/network_inventory.json
✅ CSV  report saved : /content/network_lab/network_inventory.csv
✅ TXT  report saved : /content/network_lab/network_inventory_summary.txt

📁 All files in lab directory:
   audit/configs/rtr-core-01_config.txt          0.74 KB
   audit/configs/sw-admin-01_config.txt          0.72 KB
   audit/configs/sw-ict-01_config.txt            0.71 KB
   configs/rtr-core-01_config.txt                0.74 KB
   configs/sw-admin-01_config.txt                0.72 KB
   configs/sw-ict-01_config.txt                  0.71 KB
   network_inventory.csv                         0.34 KB
   network_inventory.json                        0.8 KB
   network_inventory_summary.txt                 0.66 KB
   sw-admin-01_ports_config.txt                  0.44 KB
   uptime_log.json                               1.62 KB


---
##  Final Challenge — End-to-End Automation Script

Combine everything learned in this lab into a single function `run_network_audit(hosts, devices)` that:

1. **Resolves** all hostnames in `hosts` to IP addresses.
2. **Probes** ports 80 and 443 on each resolved host.
3. **Generates** base device configs for all devices in `devices`.
4. **Parses** the simulated interface output (`SHOW_IP_INT_BRIEF`).
5. **Detects anomalies** in the parsed interface data.
6. **Exports** a full inventory report (JSON + CSV + TXT).
7. Prints a clean **Audit Summary** at the end showing counts for each step.

Your function should run without errors and produce output files in `LAB_DIR / "audit"`.

In [29]:
# Your final challenge solution here

def run_network_audit(hosts, devices):
    """
    End-to-end network automation audit.
    """
    audit_dir = LAB_DIR / "audit"
    audit_dir.mkdir(exist_ok=True)

    print("\n" + "="*60)
    print("🚀 NETWORK AUDIT STARTING")
    print("="*60)

    # ── Step 1: Resolve hostnames ──────────────────────────────
    print("\n📡 STEP 1: Resolving Hostnames...")
    dns_map = resolve_hostnames(hosts)
    resolved = [h for h, ip in dns_map.items() if ip is not None]

    # ── Step 2: Probe ports 80 & 443 ──────────────────────────
    print("\n🔎 STEP 2: Probing Ports 80 & 443...")
    web_ports = {80: "HTTP", 443: "HTTPS"}
    port_results = {}
    for host in resolved:
        open_p = scan_host(host, web_ports)
        port_results[host] = open_p

    # ── Step 3: Generate device configs ───────────────────────
    print("\n⚙️  STEP 3: Generating Device Configurations...")
    cfg_dir = audit_dir / "configs"
    cfg_dir.mkdir(exist_ok=True)
    manifest = []
    for device in devices:
        config = generate_base_config(device)
        filename = f"{device['hostname'].lower()}_config.txt"
        filepath = cfg_dir / filename
        with open(filepath, "w") as f:
            f.write(config)
        manifest.append({"device": device["hostname"], "file": filename})
        print(f"  ✅ Config saved: {filename}")

    # ── Step 4: Parse simulated interface output ───────────────
    print("\n📋 STEP 4: Parsing Interface Output...")
    parsed_intfs = parse_show_ip_int_brief(SHOW_IP_INT_BRIEF)
    up   = sum(1 for i in parsed_intfs if i["status"] == "up")
    down = len(parsed_intfs) - up
    print(f"  Interfaces found: {len(parsed_intfs)}  |  🟢 UP: {up}  🔴 DOWN: {down}")

    # ── Step 5: Detect anomalies ───────────────────────────────
    print("\n🔍 STEP 5: Detecting Anomalies...")
    anomalies = detect_anomalies(parsed_intfs)

    # ── Step 6: Export inventory report ───────────────────────
    print("\n📦 STEP 6: Exporting Inventory Report...")
    health_results = check_website_health([f"https://{h}" for h in resolved])
    inventory = build_inventory(devices, parsed_intfs, health_results)
    export_inventory(inventory, audit_dir)

    # ── Audit Summary ──────────────────────────────────────────
    print("\n" + "="*60)
    print("✅ AUDIT COMPLETE — SUMMARY")
    print("="*60)
    print(f"  Hosts resolved       : {len(resolved)}/{len(hosts)}")
    print(f"  Hosts with port 443  : {sum(1 for p in port_results.values() if 443 in p)}")
    print(f"  Configs generated    : {len(manifest)}")
    print(f"  Interfaces parsed    : {len(parsed_intfs)}")
    print(f"  Anomalies detected   : {len(anomalies)}")
    print(f"  Inventory records    : {len(inventory)}")
    print(f"  Output directory     : {audit_dir}")
    print("="*60)


# Run it
run_network_audit(
    hosts=["google.com", "github.com", "wikipedia.org"],
    devices=DEVICES
)

def run_network_audit(hosts, devices):
    """
    End-to-end network automation audit.
    """
    audit_dir = LAB_DIR / "audit"
    audit_dir.mkdir(exist_ok=True)

    # TODO: implement each step
    pass


run_network_audit(
    hosts=["google.com", "github.com", "wikipedia.org"],
    devices=DEVICES
)


🚀 NETWORK AUDIT STARTING

📡 STEP 1: Resolving Hostnames...
Hostname                            IP Address           Status
-----------------------------------------------------------------
google.com                          64.233.179.139       ✅ Resolved
github.com                          140.82.112.3         ✅ Resolved
wikipedia.org                       208.80.153.224       ✅ Resolved

🔎 STEP 2: Probing Ports 80 & 443...

🔎 Port Scan: google.com
Port     Service      Status
--------------------------------
80       HTTP         🟢 OPEN
443      HTTPS        🟢 OPEN

  Open ports found: [80, 443]

🔎 Port Scan: github.com
Port     Service      Status
--------------------------------
80       HTTP         🟢 OPEN
443      HTTPS        🟢 OPEN

  Open ports found: [80, 443]

🔎 Port Scan: wikipedia.org
Port     Service      Status
--------------------------------
80       HTTP         🟢 OPEN
443      HTTPS        🟢 OPEN

  Open ports found: [80, 443]

⚙️  STEP 3: Generating Device Configu

---
## Reflection Questions

Answer each question below in 2 – 4 sentences by editing this Markdown cell.

**Q1.** What is the difference between the `ipaddress.ip_address()` and `ipaddress.ip_network()` objects? When would you use each?

> ip_address() represents a single address eg. 192.168.1.10 and is used when you need to review or compare one address. ip_network() represents a range of addresses like 192.168.1.0/24 and is used when you need subnet information — like the broadcast address, usable hosts, or checking membership. You'd use ip_address() to validate user input and ip_network() to plan or audit subnets.



---

**Q2.** A port probe returns `CLOSED` for port 22 (SSH) on a switch. List **two possible reasons** for this result (not necessarily that SSH is disabled).

>1.firewall or ACL (Access Control List) between the probing machine and the switch may be blocking port 22 even if SSH is enabled on the device. Second.
2. The switch may be configured to only allow SSH connections from specific management IP addresses (using an access-class on the VTY lines), which may cause the connection to be refused from any other source.

---

**Q3.** Why is template-based config generation preferred over manually writing device configurations, especially in a network with 200+ devices?

> Template-based generation eliminates human error where every device gets a consistent, correctly formatted configuration derived from the same source of truth. It also saves you enormous time; what would take days of manual typing across 200 devices can be done in seconds with a script. Additionally, when a policy changes (e.g., a new NTP server), you can update the template once and regenerate the configurations instantly rather than editing each device individually.

---

**Q4.** An interface shows `status: up` but `protocol: down`. What does this typically indicate in a real Cisco network? What would you check first?

> It means Layer 1 (the physical link) is working — the cable is connected and the port detects a signal — but Layer 2 (the data link) has failed to come up. This is commonly caused by an encapsulation mismatch, a missing or mismatched keepalive, or a duplex/speed mismatch between the two ends. The first thing to check would be the encapsulation settings on both sides of the link using show interfaces and confirm both ends agree on protocol settings

---

**Q5.** Name **two Python libraries** not used in this lab that are commonly used in production network automation, and briefly explain what each does.

> Netmiko is a multi-vendor SSH library thta is built on top of Paramiko ;it simplifies connecting to and running commands on network devices like Cisco, Juniper, and Arista routers and switches . It handles the quirks of each vendor's CLI automatically. NAPALM (Network Automation and Programmability Abstraction Layer with Multivendor support) provides a unified API across different network operating systems,  you can retrieve interface status, push configs, or compare configuration diffs using the same Python code regardless of whether the device runs IOS, JunOS, or EOS.

---
##  Lab Summary

| Part | Topic | Key Library / Tool |
|------|-------|--------------------|
| 1 | IP address & subnet automation | `ipaddress` |
| 2 | DNS resolution & port probing | `socket` |
| 3 | HTTP health checks & REST API queries | `requests` |
| 4 | Device config generation from templates | Python f-strings & `pathlib` |
| 5 | Parsing network device output | `re` (regex) |
| 6 | Inventory report generation | `json`, `csv` |

### What's Next?
- **Netmiko / Paramiko** — SSH into real or virtual routers/switches (GNS3, Cisco DevNet Sandbox)
- **NAPALM** — Vendor-agnostic network automation library
- **Nornir** — Python automation framework for large-scale network operations
- **Ansible** — Agentless IT automation, widely used for network device management
- **REST APIs** — Cisco DNA Center, Meraki Dashboard, Juniper Mist, etc.

---
*CNS 4107: Special Topics in Computer Networks-Network Automation — Strathmore University*